In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata'] 
coleccion = db['AutoTec'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [9]:
import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- LIMPIEZA ---
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")

# --- CONFIGURACIÓN ---
NOMBRE_GRUPO = "AutoTec"
URL_OBJETIVO = "https://www.newkar.cl/shop?order=publish_date+desc&tags=136"
datos_finales = []

options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--headless")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

try:
    driver = webdriver.Chrome(options=options)
    print("🚀 Navegador iniciado.")
    driver.get(URL_OBJETIVO)

    # Esperamos a que los productos sean visibles
    WebDriverWait(driver, 20).until(
        EC.visibility_of_any_elements_located((By.CSS_SELECTOR, "div.oe_product"))
    )
    
    # Scroll para asegurar carga de datos
    driver.execute_script("window.scrollTo(0, 1000);")
    time.sleep(3)

    bloques = driver.find_elements(By.CSS_SELECTOR, "div.oe_product")
    print(f"--- Procesando {len(bloques)} bloques encontrados ---")

    for i, bloque in enumerate(bloques):
        try:
            # SELECTOR DE NOMBRE REFORZADO:
            # Intenta buscar por la clase de Odoo, y si falla, busca cualquier enlace en el div
            try:
                nombre_el = bloque.find_element(By.CSS_SELECTOR, "h6.o_wsale_products_item_title a")
            except:
                nombre_el = bloque.find_element(By.CSS_SELECTOR, "div.o_wsale_product_information a")
            
            nombre = nombre_el.text.strip()

            # SELECTOR DE PRECIO REFORZADO:
            # Busca la clase 'oe_currency_value' o cualquier span con texto numérico
            try:
                precio_raw = bloque.find_element(By.CSS_SELECTOR, "span.oe_currency_value").text.strip()
            except:
                precio_raw = "0"

            # Limpieza total de caracteres no numéricos
            precio_numerico = re.sub(r'[^\d]', '', precio_raw)
            precio_final = float(precio_numerico) if precio_numerico else 0.0

            if nombre:
                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio_final,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
                # Opcional: print para ver avance en consola
                # print(f"Coche {i+1}: {nombre} -> {precio_final}")

        except Exception:
            continue

    print(f"✅ Extracción terminada: {len(datos_finales)} autos listos.")

finally:
    if driver:
        driver.quit()

# --- GUARDAR EN MONGODB ---
try:
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["proyecto_bigdata"]
    coleccion = db["AutoTec"]

    if datos_finales:
        coleccion.insert_many(datos_finales)
        print(f"💾 {len(datos_finales)} registros guardados en la colección AutoTec.")
    else:
        print("❌ No se obtuvieron datos. Verifica si el sitio tiene bloqueos.")

except Exception as e:
    print(f"❌ Error MongoDB: {e}")

🚀 Navegador iniciado.
--- Procesando 26 bloques encontrados ---
✅ Extracción terminada: 26 autos listos.
💾 26 registros guardados en la colección AutoTec.
